<h1 align="center">Join Submission and Numeric Data Sets to Find Numeric Facts</h1>

### Import Libraries

In [ ]:
import polars as pl
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

%matplotlib inline
display(HTML("<style>.container { width:100% !important; }</style>"))

### Read Submission Data Set for 2022 Q1 from [Financial Statement Data Sets](https://www.sec.gov/dera/data/financial-statement-data-sets)

In [ ]:
submissionsQ1 = pl.read_csv('data/2022q1/sub.txt', separator='\t', infer_schema_length=10000)

### Look at the contents of submissionsQ1 DataFrame. We have 7,237 submissions with 36 columns.

In [ ]:
submissionsQ1

### Read Numeric Data Set for 2022 Q1 — 3,264,632 facts, lazy-loaded for performance

In [ ]:
# scan_csv stays lazy; collect() only when needed
numericFactsQ1 = (
    pl.scan_csv('data/2022q1/num.txt', separator='\t', infer_schema_length=10000)
    .collect()
)

### Look at contents of numericFactsQ1 DataFrame

In [ ]:
numericFactsQ1

### Join numericFactsQ1 DataFrame with submissionsQ1 DataFrame using adsh as the join key

In [ ]:
joinedFactsAndSubmissionsQ1 = numericFactsQ1.join(submissionsQ1, on='adsh', how='left')

### 3,264,632 Joined Records for 2022 Q1

In [ ]:
joinedFactsAndSubmissionsQ1

### Filter for Annual Revenues for United Airlines using RevenueFromContractWithCustomerExcludingAssessedTax tag

In [ ]:
revenueFactsForUnitedAirlines = (
    joinedFactsAndSubmissionsQ1
    .filter(
        (pl.col('name') == 'UNITED AIRLINES HOLDINGS, INC.')
        & (pl.col('tag') == 'RevenueFromContractWithCustomerExcludingAssessedTax')
        & (pl.col('qtrs') == 4)
        & (pl.col('coreg') == 'UnitedAirLinesInc')
    )
    .select(['name', 'tag', 'ddate', 'qtrs', 'uom', 'value'])
)
revenueFactsForUnitedAirlines

### Scale Revenue to Billions and Create a New Column Named valueInBillions

In [ ]:
revenueFactsForUnitedAirlines = (
    revenueFactsForUnitedAirlines
    .with_columns((pl.col('value') / 1_000_000_000).alias('valueInBillions'))
    .sort('ddate')
)
revenueFactsForUnitedAirlines

### Create a Bar Chart for Annual Revenue of United Airlines

In [ ]:
revenueFactsForUnitedAirlines.to_pandas().plot(
    x='ddate', y='valueInBillions', kind='bar', figsize=(12,6), color='orange'
)
plt.title('United Airlines Annual Revenue', fontsize=20)
plt.xlabel('12 Months Ended', fontsize=15)
plt.ylabel('Total Revenue (in billions)', fontsize=15)
plt.ticklabel_format(axis='y', style='plain')
plt.xticks(rotation=0)
plt.yticks(np.arange(0, 51, step=5))
plt.savefig('images/UnitedAirlinesRevenue.jpg')
plt.show()

### Filter for Annual Revenue of American Airlines and Scale to Billions

In [ ]:
revenueFactsForAmerican = (
    joinedFactsAndSubmissionsQ1
    .filter(
        (pl.col('name') == 'AMERICAN AIRLINES GROUP INC.')
        & (pl.col('tag') == 'RevenueFromContractWithCustomerExcludingAssessedTax')
        & (pl.col('qtrs') == 4)
        & (pl.col('coreg') == 'AmericanAirlinesInc')
    )
    .select(['name', 'tag', 'ddate', 'qtrs', 'uom', 'value'])
    .with_columns((pl.col('value') / 1_000_000_000).alias('valueInBillions'))
    .sort('ddate')
)
revenueFactsForAmerican

### Combine Revenue DataFrames for Both Airlines

In [ ]:
revenueFacts = pl.concat([revenueFactsForUnitedAirlines, revenueFactsForAmerican]).sort('ddate')
revenueFacts

### Visualize Revenue for Both Airlines

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(x='ddate', y='valueInBillions', hue='name', data=revenueFacts.to_pandas())
plt.title('Annual Revenue', fontsize=20)
plt.xlabel('12 Months Ended', fontsize=15)
plt.ylabel('Total Revenue (in billions)', fontsize=15)
plt.ticklabel_format(axis='y', style='plain')
plt.xticks(rotation=0)
plt.yticks(np.arange(0, 51, step=5))
plt.savefig('images/RevenueBenchmarking.jpg')
plt.show()

### Filter for Income Tax for Both Airlines using IncomeTaxExpenseBenefit tag

In [ ]:
taxFacts = pl.concat([
    joinedFactsAndSubmissionsQ1
    .filter(
        (pl.col('name') == 'UNITED AIRLINES HOLDINGS, INC.')
        & (pl.col('tag') == 'IncomeTaxExpenseBenefit')
        & (pl.col('qtrs') == 4)
        & (pl.col('coreg') == 'UnitedAirLinesInc')
    )
    .select(['name', 'qtrs', 'ddate', 'uom', 'value'])
    .with_columns((pl.col('value') / 1_000_000).alias('valueInMillions')),
    joinedFactsAndSubmissionsQ1
    .filter(
        (pl.col('name') == 'AMERICAN AIRLINES GROUP INC.')
        & (pl.col('tag') == 'IncomeTaxExpenseBenefit')
        & (pl.col('qtrs') == 4)
        & (pl.col('coreg') == 'AmericanAirlinesInc')
    )
    .select(['name', 'qtrs', 'ddate', 'uom', 'value'])
    .with_columns((pl.col('value') / 1_000_000).alias('valueInMillions'))
]).sort('ddate')
taxFacts

### Visualize Income Tax for Both Airlines

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(x='ddate', hue='name', y='valueInMillions', data=taxFacts.to_pandas())
plt.title('Income Tax Expense (Benefit)', fontsize=20)
plt.xlabel('12 Months Ended', fontsize=15)
plt.ylabel('Income Tax Expense (Benefit) in millions', fontsize=15)
plt.ticklabel_format(axis='y', style='plain')
plt.xticks(rotation=0)
plt.yticks(np.arange(-2500, 1050, step=500))
plt.savefig('images/TaxBenchmarking.jpg')
plt.show()